In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
# ============================================================
# AMAZON ML CHALLENGE 2026
# Business Entity Resolution
# Configuration
# ============================================================

from pathlib import Path
import os
import re
import gc
import time
import unicodedata
import pandas as pd
import numpy as np

# CHANGE THIS to the exact Kaggle dataset path shown in your notebook.
DATA_DIR = Path("/kaggle/input/datasets/shreenithib/amazon-ml-challenge")

TRAIN_DIR = DATA_DIR / "dataset" / "train"
TEST_DIR  = DATA_DIR / "dataset" / "test"

OUTPUT_DIR = Path("/kaggle/working/amazon_ml_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Train directory:", TRAIN_DIR)
print("Test directory :", TEST_DIR)
print("Output directory:", OUTPUT_DIR)

Train directory: /kaggle/input/datasets/shreenithib/amazon-ml-challenge/dataset/train
Test directory : /kaggle/input/datasets/shreenithib/amazon-ml-challenge/dataset/test
Output directory: /kaggle/working/amazon_ml_output


In [4]:
# ============================================================
# VERIFY DATASET
# ============================================================

expected_files = [
    TRAIN_DIR / "train_source1.tsv",
    TRAIN_DIR / "train_source2.tsv",
    TRAIN_DIR / "train_source3.tsv",
    TRAIN_DIR / "train_ground_truth.tsv",
    TEST_DIR / "test_source1.tsv",
    TEST_DIR / "test_source2.tsv",
    TEST_DIR / "test_source3.tsv",
]

for path in expected_files:
    print(
        f"{'FOUND':6} | {path} | "
        f"{path.stat().st_size / (1024**3):.2f} GB"
        if path.exists()
        else f"{'MISSING':6} | {path}"
    )

MISSING | /kaggle/input/datasets/shreenithib/amazon-ml-challenge/dataset/train/train_source1.tsv
MISSING | /kaggle/input/datasets/shreenithib/amazon-ml-challenge/dataset/train/train_source2.tsv
MISSING | /kaggle/input/datasets/shreenithib/amazon-ml-challenge/dataset/train/train_source3.tsv
MISSING | /kaggle/input/datasets/shreenithib/amazon-ml-challenge/dataset/train/train_ground_truth.tsv
MISSING | /kaggle/input/datasets/shreenithib/amazon-ml-challenge/dataset/test/test_source1.tsv
MISSING | /kaggle/input/datasets/shreenithib/amazon-ml-challenge/dataset/test/test_source2.tsv
MISSING | /kaggle/input/datasets/shreenithib/amazon-ml-challenge/dataset/test/test_source3.tsv


In [6]:
from pathlib import Path

# Your actual Kaggle dataset location
DATA_DIR = Path("/kaggle/input/datasets/shreenithib/amazon-ml-challenge")

TRAIN_DIR = DATA_DIR / "dataset" / "train"
TEST_DIR  = DATA_DIR / "dataset" / "test"

OUTPUT_DIR = Path("/kaggle/working/amazon_ml_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR :", DATA_DIR)
print("TRAIN_DIR:", TRAIN_DIR)
print("TEST_DIR :", TEST_DIR)

DATA_DIR : /kaggle/input/datasets/shreenithib/amazon-ml-challenge
TRAIN_DIR: /kaggle/input/datasets/shreenithib/amazon-ml-challenge/dataset/train
TEST_DIR : /kaggle/input/datasets/shreenithib/amazon-ml-challenge/dataset/test


In [7]:
from pathlib import Path

# Actual Kaggle dataset location
DATA_DIR = Path("/kaggle/input/datasets/shreenithib/amazon-ml-challenge")

# Files are directly inside DATA_DIR
TRAIN_DIR = DATA_DIR
TEST_DIR = DATA_DIR

# Writable output directory
OUTPUT_DIR = Path("/kaggle/working/amazon_ml_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR :", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

DATA_DIR : /kaggle/input/datasets/shreenithib/amazon-ml-challenge
OUTPUT_DIR: /kaggle/working/amazon_ml_output


In [9]:
import pandas as pd

sample = pd.read_csv(
    DATA_DIR / "train_source1.tsv",
    sep="\t",
    nrows=10
)

print(sample)
print("\nColumns:")
print(sample.columns.tolist())

      entity_id                             business_name  \
0  S1-925783039                       Orelee's Barbershop   
1  S1-773889195                               Prime Money   
2  S1-377745466                             B+ Retail Inc   
3  S1-133037285                             Christ Chapel   
4  S1-755362802                   Prabhav Business Center   
5  S1-851869949                Custom Wealth Services LLC   
6  S1-785847572  Consulting Nyasa Nursing Private Limited   
7   S1-27541239                         Nexus Anchor Rain   
8  S1-629417405                         Moore Bitwise Inc   
9   S1-22305073                Dermatology Green Medicine   

                                    business_address country  
0             1795 Westchester Drive, High Point, NC      US  
1                    17560 Ellis Road, Tahlequah, OK      US  
2                1712 Montebello Avenue, Phoenix, AZ      US  
3  2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD      US  
4  797, Lake 

In [8]:
expected_files = [
    DATA_DIR / "train_source1.tsv",
    DATA_DIR / "train_source2.tsv",
    DATA_DIR / "train_source3.tsv",
    DATA_DIR / "train_ground_truth.tsv",
    DATA_DIR / "test_source1.tsv",
    DATA_DIR / "test_source2.tsv",
    DATA_DIR / "test_source3.tsv",
]

for path in expected_files:
    if path.exists():
        size_gb = path.stat().st_size / (1024**3)
        print(f"FOUND   | {path.name:25s} | {size_gb:.2f} GB")
    else:
        print(f"MISSING | {path}")

FOUND   | train_source1.tsv         | 0.20 GB
FOUND   | train_source2.tsv         | 0.46 GB
FOUND   | train_source3.tsv         | 0.47 GB
FOUND   | train_ground_truth.tsv    | 0.12 GB
FOUND   | test_source1.tsv          | 0.16 GB
FOUND   | test_source2.tsv          | 0.47 GB
FOUND   | test_source3.tsv          | 0.47 GB


In [1]:
import re
import unicodedata
import pandas as pd
import numpy as np
import gc
import time
from collections import defaultdict

# ------------------------------------------------------------
# TEXT NORMALIZATION
# ------------------------------------------------------------

LEGAL_SUFFIX_MAP = {
    "incorporated": "inc",
    "corporation": "corp",
    "company": "co",
    "limited": "ltd",
    "private": "pvt",
    "llc": "llc",
    "llp": "llp",
}


def normalize_unicode(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()

    return text


def normalize_name(text):
    text = normalize_unicode(text)

    # Replace punctuation with spaces
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    # Standardize legal suffixes
    tokens = [
        LEGAL_SUFFIX_MAP.get(token, token)
        for token in tokens
    ]

    return " ".join(tokens)


def normalize_address(text):
    text = normalize_unicode(text)

    replacements = {
        r"\bstreet\b": "st",
        r"\broad\b": "rd",
        r"\bavenue\b": "ave",
        r"\blane\b": "ln",
        r"\bdrive\b": "dr",
        r"\bhighway\b": "hwy",
        r"\bparkway\b": "pkwy",
        r"\bboulevard\b": "blvd",
    }

    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)

    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()

    return text


print(normalize_name("ABC Corporation Pvt. Ltd."))
print(normalize_address("500 Market Street, San Jose"))

abc corp pvt ltd
500 market st san jose


In [2]:
# ============================================================
# STEP 2: BUILD SOURCE 2 NAME + COUNTRY INDEX
# ============================================================

def build_name_country_index(file_path, chunk_size=250_000):

    index = defaultdict(list)

    start = time.time()
    total_rows = 0

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        usecols=[
            "entity_id",
            "business_name",
            "country"
        ],
        chunksize=chunk_size
    ):

        chunk["name_norm"] = (
            chunk["business_name"]
            .map(normalize_name)
        )

        for row in chunk.itertuples(index=False):

            if not row.name_norm:
                continue

            key = (
                row.name_norm,
                row.country
            )

            index[key].append(row.entity_id)

        total_rows += len(chunk)

        print(
            f"Processed {total_rows:,} rows",
            end="\r"
        )

        del chunk
        gc.collect()

    elapsed = (time.time() - start) / 60

    print()
    print(f"Processed: {total_rows:,} rows")
    print(f"Unique name+country keys: {len(index):,}")
    print(f"Time: {elapsed:.2f} minutes")

    return index


print("Building Source 2 index...")

s2_index = build_name_country_index(
    DATA_DIR / "train_source2.tsv"
)

Building Source 2 index...


NameError: name 'DATA_DIR' is not defined

In [4]:
from pathlib import Path

# Your actual Kaggle dataset location
DATA_DIR = Path("/kaggle/input/datasets/shreenithib/amazon-ml-challenge")

print("Dataset exists:", DATA_DIR.exists())
print("Dataset path:", DATA_DIR)

Dataset exists: True
Dataset path: /kaggle/input/datasets/shreenithib/amazon-ml-challenge


In [4]:
# ============================================================
# STEP 2: BUILD SOURCE 2 NAME + COUNTRY INDEX
# ============================================================

def build_name_country_index(file_path, chunk_size=250_000):

    index = defaultdict(list)

    start = time.time()
    total_rows = 0

    for chunk in pd.read_csv(
        file_path,
        sep="\t",
        usecols=[
            "entity_id",
            "business_name",
            "country"
        ],
        chunksize=chunk_size
    ):

        chunk["name_norm"] = (
            chunk["business_name"]
            .map(normalize_name)
        )

        for row in chunk.itertuples(index=False):

            if not row.name_norm:
                continue

            key = (
                row.name_norm,
                row.country
            )

            index[key].append(row.entity_id)

        total_rows += len(chunk)

        print(
            f"Processed {total_rows:,} rows",
            end="\r"
        )

        del chunk
        gc.collect()

    elapsed = (time.time() - start) / 60

    print()
    print(f"Processed: {total_rows:,} rows")
    print(f"Unique name+country keys: {len(index):,}")
    print(f"Time: {elapsed:.2f} minutes")

    return index


print("Building Source 2 index...")

s2_index = build_name_country_index(
    DATA_DIR / "train_source2.tsv"
)


Building Source 2 index...
Processed 5,034,616 rows
Processed: 5,034,616 rows
Unique name+country keys: 3,971,627
Time: 1.15 minutes


In [5]:
# ============================================================
# STEP 3: BUILD SOURCE 3 NAME + COUNTRY INDEX
# ============================================================

print("Building Source 3 index...")

s3_index = build_name_country_index(
    DATA_DIR / "train_source3.tsv"
)

Building Source 3 index...
Processed 5,285,603 rows
Processed: 5,285,603 rows
Unique name+country keys: 4,223,574
Time: 1.43 minutes


In [6]:
# ============================================================
# STEP 4: INSPECT GROUND TRUTH
# ============================================================

GT_PATH = DATA_DIR / "train_ground_truth.tsv"

gt_sample = pd.read_csv(
    GT_PATH,
    sep="\t",
    nrows=10
)

print("Ground truth columns:")
print(gt_sample.columns.tolist())

print("\nShape of sample:", gt_sample.shape)

display(gt_sample)

Ground truth columns:
['source1_entity_id', 'matched_entity_ids']

Shape of sample: (10, 2)


,source1_entity_id,matched_entity_ids
0,S1-965667,"S2-681193310,S2-743505751,S3-775321672,S3-1129..."
1,S1-55344266,"S2-249013014,S2-197070651,S3-478195123,S3-3843..."
2,S1-343815751,"S2-790675320,S2-479876582,S3-878454467"
3,S1-656753428,"S2-153058913,S2-24659151,S3-679606215"
4,S1-102811957,"S2-478959098,S2-553508714,S2-625774905,S3-7280..."
5,S1-18727616,"S2-755677256,S3-187831601,S3-641489370,S3-4762..."
6,S1-318373630,"S2-660036492,S3-804600254"
7,S1-86989137,"S3-274817120,S3-312496301"
8,S1-29845983,"S2-648035184,S3-588502663"
9,S1-789009573,"S2-383871912,S3-74481402,S3-576451439"


In [1]:
# ============================================================
# STEP 5: LOAD 10,000 SOURCE 1 RECORDS
# ============================================================

S1_SAMPLE_N = 10_000

s1_sample = pd.read_csv(
    DATA_DIR / "train_source1.tsv",
    sep="\t",
    nrows=S1_SAMPLE_N,
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

s1_sample["name_norm"] = s1_sample["business_name"].map(normalize_name)

print(f"Loaded Source 1 records: {len(s1_sample):,}")

display(s1_sample.head())

NameError: name 'pd' is not defined

In [5]:
import pandas as pd
# ============================================================
# STEP 5: LOAD 10,000 SOURCE 1 RECORDS
# ============================================================

S1_SAMPLE_N = 10_000

s1_sample = pd.read_csv(
    DATA_DIR / "train_source1.tsv",
    sep="\t",
    nrows=S1_SAMPLE_N,
    usecols=[
        "entity_id",
        "business_name",
        "business_address",
        "country"
    ]
)

s1_sample["name_norm"] = s1_sample["business_name"].map(normalize_name)

print(f"Loaded Source 1 records: {len(s1_sample):,}")

display(s1_sample.head())

NameError: name 'normalize_name' is not defined

In [6]:
# ============================================================
# STEP 4.5: RE-INITIALIZE NOTEBOOK IMPORTS
# ============================================================

from pathlib import Path
import re
import unicodedata
import pandas as pd
import numpy as np
import gc
import time
from collections import defaultdict

DATA_DIR = Path("/kaggle/input/datasets/shreenithib/amazon-ml-challenge")
OUTPUT_DIR = Path("/kaggle/working/amazon_ml_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Environment initialized successfully.")
print("DATA_DIR:", DATA_DIR)
Then check that our indexes still exist

Run:

# ============================================================
# CHECK EXISTING INDEXES
# ============================================================

print("s2_index exists:", "s2_index" in globals())
print("s3_index exists:", "s3_index" in globals())
print("normalize_name exists:", "normalize_name" in globals())

SyntaxError: invalid syntax (2326739575.py, line 20)